In [ ]:
# %pip install nlpaug

import pandas as pd
import nltk
import nlpaug.augmenter.word as naw
import nlpaug.augmenter.char as nac 
from nlpaug.flow import Sometimes

nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger_eng')

# Some constants
dataset_path = "../resources/dataset"
dataset = 'youtoxic_english_1000.csv'
augmented_dataset = f"synonym_{dataset}"
original_dataset = f"{dataset_path}/{dataset}"

# Array of columns to use for augmenting dataset
targets = [
    'IsToxic', 
    'IsAbusive', 
    'IsProvocative', 
    'IsObscene', 
    'IsHatespeech', 
    'IsRacist',
    # 'IsThreat',
    # 'IsReligiousHate',
    # 'IsNationalist'
]

# Load original dataset
df = pd.read_csv(original_dataset)
df_work = df.drop_duplicates(subset=['Text'])

def augment_text_data(df, text_column='Text', output_categories=None, num_augments=3):
    """
    Augments the text data in a DataFrame using Synonym Replacement, 
    keeping ALL original records.
    """
    if output_categories is None:
        output_categories = [
            'IsToxic', 'IsAbusive', 'IsProvocative', 'IsObscene', 
            'IsHatespeech', 'IsRacist'
        ]

    synonym_aug = naw.SynonymAug(aug_min=1, aug_p=0.5)

    
    augmented_data = []
    positive_count = 0

    print(f"Starting augmentation on {len(df)} records...")
    
    # Iterate through the DataFrame rows
    for index, row in df.iterrows():
        # --- FIX 1: Add ALL original records first ---
        # Keep the original record regardless of its labels
        augmented_data.append(row.copy()) 

        # Check if one of the selected categories is true
        is_positive = row[output_categories].sum() > 0
        
        # --- FIX 2: Only proceed to augment if it's a positive record ---
        if is_positive:
            positive_count += 1
            original_text = row[text_column]
            
            # Generate the specified number of augmented texts
            for i in range(num_augments):
                # The augment() method takes a string and returns a list of augmented strings
                augmented_texts = synonym_aug.augment(original_text, n=1)
                
                if augmented_texts:
                    augmented_text = augmented_texts[0]
                    
                    # Create a new record with the augmented text
                    new_row = row.copy()
                    new_row[text_column] = augmented_text
                    
                    # The labels remain the same for the augmented text (Positive)
                    augmented_data.append(new_row)

    # Concatenate all data into a final DataFrame
    final_df = pd.DataFrame(augmented_data)
    
    # Adjusted print statement for clarity
    total_original_records = len(df)
    total_new_records = positive_count * num_augments
    
    print("-" * 50)
    print(f"Total Original Records Kept: {total_original_records}")
    print(f"Positive Records Augmented: {positive_count}")
    print(f"New Augmented Records Added: {total_new_records}")
    print(f"Final Dataset Size: {len(final_df)} records.")
    print("-" * 50)
    
    return final_df

augmented_df = augment_text_data(
    df_work, 
    text_column='Text', 
    output_categories=targets, 
    num_augments=4
)

final_df = augmented_df.drop_duplicates(subset=['Text'])

print(f"\nTotal Records augmented:      {len(augmented_df)}")
print(f"\nTotal Records after cleaning: {len(final_df)}")

# Save new dataset
augmented_df.to_csv(f"{dataset_path}/{augmented_dataset}")

[nltk_data] Downloading package wordnet to /home/vscode/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/vscode/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/vscode/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


Starting augmentation on 997 records...
--------------------------------------------------
Total Original Records Kept: 997
Positive Records Augmented: 459
New Augmented Records Added: 1836
Final Dataset Size: 2833 records.
--------------------------------------------------

Total Records augmented:      2833

Total Records after cleaning: 2735
